In [1]:
# Community-verification/util/io.py をimportするためのコード
# io.pyを編集した場合に、Jupyter Notebook上で再読み込みするために必要
import importlib
import os
import sys
sys.path.append(os.path.abspath("..")) 
from util import io
importlib.reload(io)

<module 'util.io' from '/Users/kaoriogawa/研究/community-detection-social-networks-dev/Community-verification/util/io.py'>

In [ ]:
import networkx as nx
from pathlib import Path

name = "LFR01"

path = Path("../../data") / name

# データ読み込み
G = nx.read_edgelist(path / "edgelist.txt.gz", create_using=nx.DiGraph(), nodetype=int)
communities = io.communitydict_to_communitylist(io.read_communitydict(path / "communities.txt.gz"))

In [ ]:
import json
import random
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple, Union

import networkx as nx


# 1. 共通ユーティリティ & 真情報シード選定

def get_subgraph_and_node_set(
    G: nx.DiGraph, community_nodes: List[int]) -> Tuple[nx.DiGraph, set]:
    """コミュニティノード集合と、それらに基づく誘発部分グラフを取得する helper 関数。"""
    nodes_set = set(community_nodes)
    sub_G = G.subgraph(nodes_set)
    return sub_G, nodes_set


# 真情報の発信源（コミュニティID: 2）の中心ノードを選定
def select_true_seed_users(
    G: nx.DiGraph,
    community_nodes: List[int],
    top_k: int = 1,
    method: Literal["internal_degree", "pagerank"] = "internal_degree") -> List[int]:
    """
    Parameters
    ----------
    G : nx.DiGraph
        全体ネットワークグラフ。
    community_nodes : List[int]
        対象コミュニティ（真情報コミュニティ）に属するノードのIDリスト。
    top_k : int, default=1
        選定するシードノードの数。
    method : {"internal_degree", "pagerank"}, default="internal_degree"
        中心性の評価尺度。
        - "internal_degree": コミュニティ内部の総次数（In-degree + Out-degree）。
        - "pagerank": コミュニティ内誘発部分グラフでのPageRankスコア。

    Returns
    -------
    List[int]
        選定された真情報の発信源ノードIDのリスト。
    """
    sub_G, _ = get_subgraph_and_node_set(G, community_nodes)

    if method == "internal_degree":
        # 有向グラフにおけるコミュニティ内部の総次数 (入次数 + 出次数)
        scores = {node: sub_G.in_degree(node) + sub_G.out_degree(node) for node in sub_G.nodes()}
    elif method == "pagerank":
        # 部分グラフ内でのPageRankを算出
        scores = nx.pagerank(sub_G)
    else:
        raise ValueError(f"Unknown method: {method}")

    # スコアが高い順にソートして上位top_k件を返す
    sorted_nodes = sorted(scores.keys(), key=lambda n: scores[n], reverse=True)
    return sorted_nodes[:top_k]



# 2. 虚偽情報の発信源選定（各シナリオ関数）

type BridgeMode = Literal["v_only", "u_only", "both"]

# ブリッジノードに基づく虚偽情報発信源の選定
def select_bridge_seeds(
    G: nx.DiGraph,
    true_community_nodes: List[int],
    false_community_nodes: List[int],
    mode: BridgeMode = "v_only",
) -> List[int]:
    """
    自陣（Target/True）と相手国（Opponent/False）を結ぶ境界エッジ (u, v) または (v, u) を特定し、
    境界接続数が最も多いノードを選定する。
    ※ u: 自陣側ノード (True Community)
    ※ v: 相手国側ノード (False Community)

    Parameters
    ----------
    mode : {"v_only", "u_only", "both"}
        - "v_only": 相手国側の境界ハブノード v のみを選定（王道設定）。
        - "u_only": 自陣側の境界ハブノード u のみを選定（内部発生・潜入設定）。
        - "both"  : 両ノード (u, v) を同時に選定（2拠点同時発生設定）。
    """
    true_set = set(true_community_nodes)
    false_set = set(false_community_nodes)

    u_count: Dict[int, int] = {}
    v_count: Dict[int, int] = {}

    # 全体グラフのエッジから境界エッジを探す
    for u_node, v_node in G.edges():
        # Case A: True -> False (u in True, v in False)
        if u_node in true_set and v_node in false_set:
            u_count[u_node] = u_count.get(u_node, 0) + 1
            v_count[v_node] = v_count.get(v_node, 0) + 1
        # Case B: False -> True (v in False, u in True)
        elif u_node in false_set and v_node in true_set:
            v_count[u_node] = v_count.get(u_node, 0) + 1
            u_count[v_node] = u_count.get(v_node, 0) + 1

    if not u_count or not v_count:
        raise ValueError("コミュニティ間に境界エッジが見つかりませんでした。")

    # 接続数が最も多い最前線ノードを特定
    top_v = max(v_count.keys(), key=lambda n: v_count[n])
    top_u = max(u_count.keys(), key=lambda n: u_count[n])

    if mode == "v_only":
        return [top_v]
    elif mode == "u_only":
        return [top_u]
    elif mode == "both":
        return [top_v, top_u]
    else:
        raise ValueError(f"Unknown mode: {mode}")


type NodeMethod = Literal["internal_degree", "internal_pagerank", "global_pagerank"]
# コミュニティの中心ノード（最高インフルエンサー）の選定
def select_center_seeds(
    G: nx.DiGraph,
    false_community_nodes: List[int],
    method: NodeMethod = "internal_degree",
    top_k: int = 1,
) -> List[int]:
    """
    Parameters
    ----------
    method : {"internal_degree", "internal_pagerank", "global_pagerank"}
        - "internal_degree"  : コミュニティ内部の総次数（In + Out）最大ノード。
        - "internal_pagerank": コミュニティ内部グラフでのPageRank最大ノード。
        - "global_pagerank"  : グラフ全体のPageRankスコアがコミュニティ内で最大なノード。
    """
    sub_G, _ = get_subgraph_and_node_set(G, false_community_nodes)

    if method == "internal_degree":
        scores = {node: sub_G.in_degree(node) + sub_G.out_degree(node) for node in sub_G.nodes()}
    elif method == "internal_pagerank":
        scores = nx.pagerank(sub_G)
    elif method == "global_pagerank":
        global_pr = nx.pagerank(G)
        scores = {node: global_pr[node] for node in false_community_nodes}
    else:
        raise ValueError(f"Unknown method: {method}")

    sorted_nodes = sorted(scores.keys(), key=lambda n: scores[n], reverse=True)
    return sorted_nodes[:top_k]


# コミュニティの端寄り（周辺部）ノードの選定
def select_periphery_seeds(
    G: nx.DiGraph,
    false_community_nodes: List[int],
    num_seeds: int = 1,
    seed: Optional[int] = 42,
) -> List[int]:
    """
    コミュニティ内部次数（In + Out）が最も低いノード（最低1以上の結合を持つ末端ノード）
    を抽出し、その中から単一または複数のユーザーをサンプリングする。

    Parameters
    ----------
    num_seeds : int, default=1
        選定するシード数（1の場合は単一ユーザー、2以上の場合は複数ユーザーシナリオ）。
    seed : Optional[int], default=42
        乱数シード（実験の再現性確保のため）。
    """
    sub_G, _ = get_subgraph_and_node_set(G, false_community_nodes)

    # 内部次数を計算
    internal_degrees = {node: sub_G.in_degree(node) + sub_G.out_degree(node) for node in sub_G.nodes()}

    # 孤立ノード（次数0）は拡散しないため除外（内部次数 >= 1 のみ）
    valid_candidates = [node for node, deg in internal_degrees.items() if deg >= 1]

    if not valid_candidates:
        # 万が一次数1以上がない場合は全ノードを対象に
        valid_candidates = list(false_community_nodes)

    # 内部次数が最小の値を取得
    min_deg = min(internal_degrees[node] for node in valid_candidates)
    
    # 最小次数を持つ最末端ノード群を候補として抽出
    periphery_candidates = [node for node in valid_candidates if internal_degrees[node] == min_deg]

    # 指定された数だけランダムサンプリング
    if seed is not None:
        random.seed(seed)

    if len(periphery_candidates) >= num_seeds:
        selected_seeds = random.sample(periphery_candidates, num_seeds)
    else:
        # 最小次数ノード数が足りない場合は全件抽出
        selected_seeds = periphery_candidates

    return selected_seeds


# 3. JSONファイル保存パイプライン
def save_experiment_setting(
    output_dir: Path,
    scenario_name: str,
    graph_path: str,
    true_community_id: int,
    true_seed_users: List[int],
    false_seed_users: List[int],
) -> None:
    """実験設定をJSONファイルとして保存する。"""
    data = {
        "graph_path": graph_path,
        "scenario_name": scenario_name,
        "true_community_id": true_community_id,
        "true_seed_users": true_seed_users,
        "false_seed_users": false_seed_users,
    }

    output_path = output_dir / f"{scenario_name}.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"Saved: {output_path}")


# 4. 全シナリオの一括生成関数
def generate_all_scenarios(
    graph_name: str,
    true_comm_id: int, false_comm_id: int,
    priphery_multi_seeds: int, periphery_seed: int,
    ) -> None:
    # データ読み込み
    path = Path("../../data") / graph_name

    G = nx.read_edgelist(path / "edgelist.txt.gz", create_using=nx.DiGraph(), nodetype=int)
    communities = io.communitydict_to_communitylist(io.read_communitydict(path / "communities.txt.gz"))

    # コミュニティ定義
    true_nodes = communities[true_comm_id]
    false_nodes = communities[false_comm_id]

    save_setting = lambda scenario_name, false_seed_users: save_experiment_setting(
        scenario_name=scenario_name,
        false_seed_users=false_seed_users,
        graph_path=f"data/{graph_name}/edgelist.txt.gz",
        true_community_id=true_comm_id,
        # 真情報の発信源（コミュニティ2の中心ノード）を自動算出
        true_seed_users=select_true_seed_users(G, true_nodes, top_k=1, method="internal_degree"),
        # 保存ディレクトリ
        output_dir=Path("../../data/exp_settings") / graph_name / f"f_comm_{false_comm_id}",
    )

    # シナリオ別の設定リスト (ファイル名, シナリオ識別子, 抽出処理コールバック)

    import typing

    # ① ブリッジノード
    for kind in typing.get_args(BridgeMode):
        print(f"Generating bridge scenario: {kind}")
        save_setting(
            scenario_name=f"bridge_{kind}",
            false_seed_users=select_bridge_seeds(G, true_nodes, false_nodes, mode=kind),
        )

    # ② 中心ノード
    for method in typing.get_args(NodeMethod):
        print(f"Generating center scenario: {method}")
        save_setting(
            scenario_name=f"center_{method}",
            false_seed_users=select_center_seeds(G, false_nodes, method=method)
        )

    # ③ 端寄りノード
    scenarios = [
        ("periphery_single", select_periphery_seeds(G, false_nodes, num_seeds=1, seed=periphery_seed)),
        ("periphery_multiple", select_periphery_seeds(G, false_nodes, num_seeds=priphery_multi_seeds, seed=periphery_seed)),
    ]

    # シナリオを実行して保存
    for scenario_name, false_seeds in scenarios:
        save_setting(
            scenario_name=scenario_name,
            false_seed_users=false_seeds
        )

Saved: ../../data/exp_settings/LFR01/f_comm_1/bridge_v_only.json
Saved: ../../data/exp_settings/LFR01/f_comm_1/bridge_u_only.json
Saved: ../../data/exp_settings/LFR01/f_comm_1/bridge_both.json
Saved: ../../data/exp_settings/LFR01/f_comm_1/center_internal_degree.json
Saved: ../../data/exp_settings/LFR01/f_comm_1/center_pagerank.json
Saved: ../../data/exp_settings/LFR01/f_comm_1/periphery_single.json
Saved: ../../data/exp_settings/LFR01/f_comm_1/periphery_multiple.json


In [ ]:
# true_comm_id = 2
# false_comm_id = 1
generate_all_scenarios("LFR01", true_comm_id=2, false_comm_id=1, priphery_multi_seeds=5, periphery_seed=42)

In [9]:
import json
import random
import networkx as nx
import os

def create_ba_directed_graph(n:int, m:int, seed:int, output_file:str, bidirectional:bool):
    """BAモデルをベースにした有向グラフを生成し、エッジリスト（u v）形式で出力する関数

    引数:
        n (int): ノード数
        m (int): 新規ノード追加時に貼るエッジ数
        seed (int): 乱数シード
        output_file (str): 保存するテキストファイルのパス
        bidirectional (bool): Trueの場合は全エッジを双方向に、Falseの場合は確率50%で一方向化
    """

    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    # 1. BAモデルで無向グラフを作成
    g_undirected = nx.barabasi_albert_graph(n=n, m=m, seed=seed)

    # 2. 有向グラフに変換
    if bidirectional:
        g_directed = g_undirected.to_directed()
    else:
        g_directed = nx.DiGraph()
        random.seed(seed)
        for u, v in g_undirected.edges():
            if random.random() < 0.5:
                g_directed.add_edge(u, v)
            else:
                g_directed.add_edge(v, u)

    # 3. エッジリストを書き出し
    with open(output_file, "w", encoding="utf-8") as f:
        for u, v in g_directed.edges():
            f.write(f"{u} {v}\n")

    print(
        f"グラフ生成完了: ノード数={g_directed.number_of_nodes()}, エッジ数={g_directed.number_of_edges()} -> {output_file}"
    )

In [10]:
def create_config_json(
    output_path:str, graph_path:str, false_users:list[int], cost_func:str, sample_size:int, iterations:list[dict]
):
    """評価プログラム用の設定JSONファイルを生成する関数

    引数:
        output_path (str): 保存先JSONファイルのパス
        graph_path (str): 読み込むグラフ構造ファイルのパス
        false_users (list[int]): 偽情報発信源のノードIDリスト
        cost_func (str): コスト関数 ('follower' または 'influence')
        sample_size (int): 伝播シミュレーションの試行回数
        iterations (list[dict]): イテレーション条件の辞書リスト
    """
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    config_data = {
        "graph_path": graph_path,
        "false_users": false_users,
        "cost_func": cost_func,
        "sample_size": sample_size,
        "iterations": iterations,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(config_data, f, indent=2)

    print(f"設定ファイル生成完了: -> {output_path}")

In [13]:
graph_file_path = "../../data/BA01/graph_ba.txt"
config_file_path = "../../data/BA01/config.json"

# 1. 有向グラフデータの生成（引数を決定して呼び出し）
create_ba_directed_graph(
    n=10,  # ノード数
    m=2,  # エッジ生成数
    seed=42,  # シード値
    output_file="../../data/BA01/graph_ba.txt",  # 保存ファイル名
    bidirectional=False,  # ランダム一方向化
)

# 2. JSON設定ファイルの生成（引数を決定して呼び出し）
create_config_json(
    output_path="../../data/BA01/config.json",  # 出力JSON名
    graph_path="../../data/BA01/graph_ba.txt",  # 使用するグラフファイル
    false_users=[0, 1],  # 偽情報発信源
    cost_func="follower",  # コスト関数名
    sample_size=100,  # サンプル数
    iterations=[  # イテレーション条件
        {"seed": 42, "capacity": 5},
        {"seed": 43, "capacity": 10},
    ],
)

グラフ生成完了: ノード数=10, エッジ数=16 -> ../../data/BA01/graph_ba.txt
設定ファイル生成完了: -> ../../data/BA01/config.json


In [12]:
import os

# ノートブックがある場所から見たパスで作成
os.makedirs("../../data/BA01/output", exist_ok=True)

# フォルダが存在するか確認
print("outputフォルダ存在確認:", os.path.exists("../../data/BA01/output"))

outputフォルダ存在確認: True
